# 01 · SQL: perguntas de negócio
[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessicapmartins/segmentacao-clientes-olist/blob/main/notebooks/01_sql_perguntas_negocio.ipynb)

Cada pergunta vira **uma consulta SQL** e **uma frase de conclusão**. Todas as perguntas estão resolvidas, com o resultado e a conclusão de negócio logo abaixo.

**Dica de estudo:** escreva primeiro em português o que a consulta precisa fazer (de onde vem, como junta, como agrupa), depois o SQL.

Modelo das tabelas: `clientes` 1→N `pedidos` 1→N `itens_pedido` N→1 `produtos` · `pedidos` 1→N `pagamentos` · `pedidos` 1→N `avaliacoes` · `itens_pedido` N→1 `vendedores`.

In [17]:
import sqlite3, pandas as pd

try:                                    # no Colab: lê o banco do seu Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DADOS = '/content/drive/MyDrive/03_Portfolio/Projetos_ML/01_segmentacao-clientes-olist/dados'
except ModuleNotFoundError:             # fora do Colab: pasta dados/ do projeto
    DADOS = '../dados'

con = sqlite3.connect(f'{DADOS}/olist.db')

def q(sql):
    """Roda uma consulta SQL no banco da Olist e devolve um DataFrame."""
    return pd.read_sql(sql, con)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0. Conhecer as tabelas *(volte aqui sempre que tiver dúvida)*

| Tabela | Cada linha é | Colunas principais |
|---|---|---|
| `clientes` | o cadastro do cliente em um pedido | `customer_id`, `customer_unique_id` (a pessoa), `customer_state` |
| `pedidos` | um pedido | `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp` (data da compra), datas de entrega |
| `itens_pedido` | um item de um pedido | `order_id`, `product_id`, `seller_id`, **`price`**, **`freight_value`** |
| `pagamentos` | uma forma de pagamento do pedido | `order_id`, `payment_type`, `payment_installments`, `payment_value` |
| `avaliacoes` | uma avaliação | `order_id`, `review_score` (1 a 5), `review_comment_message` |
| `produtos` | um produto | `product_id`, `product_category_name` |
| `vendedores` | um vendedor | `seller_id`, `seller_state` |
| `categorias_en` | a tradução da categoria | `product_category_name`, `product_category_name_english` |

In [18]:
# DICIONÁRIO DA BASE: cada tabela, suas colunas e um exemplo de valor
for tabela in ['clientes', 'pedidos', 'itens_pedido', 'pagamentos',
               'avaliacoes', 'produtos', 'vendedores', 'categorias_en']:
    total = q(f"SELECT COUNT(*) AS n FROM {tabela}")['n'][0]
    exemplo = q(f"SELECT * FROM {tabela} LIMIT 1").T      # T = vira a tabela de lado
    exemplo.columns = ['exemplo']
    print(f"\n===== {tabela.upper()}  ({total:,} linhas) =====".replace(',', '.'))
    display(exemplo)


===== CLIENTES  (99.441 linhas) =====


,exemplo
customer_id,06b8999e2fba1a1fbc88172c00ba8bc7
customer_unique_id,861eff4711a542e4b93843c6dd7febb0
customer_zip_code_prefix,14409
customer_city,franca
customer_state,SP



===== PEDIDOS  (99.441 linhas) =====


,exemplo
order_id,e481f51cbdc54678b7cc49136f2d6af7
customer_id,9ef432eb6251297304e76186b10a928d
order_status,delivered
order_purchase_timestamp,2017-10-02 10:56:33
order_approved_at,2017-10-02 11:07:15
order_delivered_carrier_date,2017-10-04 19:55:00
order_delivered_customer_date,2017-10-10 21:25:13
order_estimated_delivery_date,2017-10-18 00:00:00



===== ITENS_PEDIDO  (112.650 linhas) =====


...

,exemplo
product_id,1e9e8ef04dbcff4541ed26657ea517e5
product_category_name,perfumaria
product_name_lenght,40.0
product_description_lenght,287.0
product_photos_qty,1.0
product_weight_g,225.0
product_length_cm,16.0
product_height_cm,10.0
product_width_cm,14.0



===== VENDEDORES  (3.095 linhas) =====


,exemplo
seller_id,3442f8959a84dea7ee197c632cb2df15
seller_zip_code_prefix,13023
seller_city,campinas
seller_state,SP



===== CATEGORIAS_EN  (71 linhas) =====


,exemplo
product_category_name,beleza_saude
product_category_name_english,health_beauty


In [19]:
# Quais tabelas existem?
q("SELECT name FROM sqlite_master WHERE type='table'")

,name
0,clientes
1,pedidos
2,itens_pedido
3,pagamentos
4,avaliacoes
5,produtos
6,vendedores
7,categorias_en
8,rfm


In [20]:
# Quais colunas uma tabela tem? Troque o nome entre parênteses.
q("PRAGMA table_info(pedidos)")[['name', 'type']]

,name,type
0,order_id,TEXT
1,customer_id,TEXT
2,order_status,TEXT
3,order_purchase_timestamp,TEXT
4,order_approved_at,TEXT
5,order_delivered_carrier_date,TEXT
6,order_delivered_customer_date,TEXT
7,order_estimated_delivery_date,TEXT


In [21]:
# Ver as primeiras linhas de uma tabela
q("SELECT * FROM itens_pedido LIMIT 5")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


**Lembrete:** o texto entre `'''` só vira consulta quando vai **dentro** de `q(...)`. Sem o `q`, o Python apenas mostra o texto de volta.

```python
q('''
SELECT ...
FROM ...
''')
```

## 1. Tamanho do negócio *(exemplo resolvido)*
Quantos pedidos, clientes únicos e vendedores existem?

In [22]:
q('''
SELECT
    (SELECT COUNT(DISTINCT order_id)           FROM pedidos)    AS pedidos,
    (SELECT COUNT(DISTINCT customer_unique_id) FROM clientes)   AS clientes_unicos,
    (SELECT COUNT(DISTINCT seller_id)          FROM vendedores) AS vendedores
''')

,pedidos,clientes_unicos,vendedores
0,99441,96096,3095


**Conclusão:** a base tem 99.441 pedidos, 96.096 clientes únicos e 3.095 vendedores. Como há menos clientes do que pedidos, alguns clientes compraram mais de uma vez (ver pergunta 7).

## 2. Faturamento mensal e crescimento
Receita (soma de `price` + `freight_value`) por mês da compra e variação % contra o mês anterior.

*Conceitos:* `strftime('%Y-%m', ...)`, CTE (`WITH`), função de janela `LAG()`.

In [23]:
q('''
WITH receita_mensal AS (                                   -- 1) receita por mês
    SELECT strftime('%Y-%m', p.order_purchase_timestamp) AS mes,
           SUM(i.price) + SUM(i.freight_value)          AS receita
    FROM pedidos p
    JOIN itens_pedido i ON p.order_id = i.order_id
    WHERE p.order_status = 'delivered'
    GROUP BY mes
)
SELECT mes,
       ROUND(receita, 2)                                  AS receita,
       ROUND(LAG(receita) OVER (ORDER BY mes), 2)         AS receita_mes_anterior,  -- 2) mês anterior
       ROUND(100.0 * (receita - LAG(receita) OVER (ORDER BY mes))
             / LAG(receita) OVER (ORDER BY mes), 1)       AS variacao_pct           -- 3) variação %
FROM receita_mensal
ORDER BY mes
''')

,mes,receita,receita_mes_anterior,variacao_pct
0,2016-09,143.46,NaN,NaN
1,2016-10,46490.66,143.46,32306.7
2,2016-12,19.62,46490.66,-100.0
3,2017-01,127482.37,19.62,649657.2
4,2017-02,271239.32,127482.37,112.8
5,2017-03,414330.95,271239.32,52.8
6,2017-04,390812.40,414330.95,-5.7
7,2017-05,566851.40,390812.40,45.0
8,2017-06,490050.37,566851.40,-13.5
9,2017-07,566299.08,490050.37,15.6


**Conclusão:** a receita dos pedidos entregues saiu de R$ 127 mil em jan/2017 para cerca de R$ 1 milhão por mês em 2018, quando se estabilizou entre R$ 0,99 mi e R$ 1,13 mi. O pico foi nov/2017 (R$ 1,15 mi, +53,7% sobre outubro), mês da Black Friday. Os meses de 2016 e as variações gigantes de dez/2016 e jan/2017 refletem dados incompletos no início da base e não devem ser lidos como crescimento real.

## 3. Top 10 categorias por receita
Use a tabela `categorias_en` para o nome em inglês e `DENSE_RANK()` para o ranking.

*Conceitos:* JOIN de 3 tabelas, `DENSE_RANK() OVER (ORDER BY ...)`.

In [24]:
q('''
WITH receita_categoria AS (
    SELECT COALESCE(c.product_category_name_english, pr.product_category_name, 'sem_categoria') AS categoria,
           SUM(i.price) + SUM(i.freight_value) AS receita,
           COUNT(DISTINCT i.order_id)          AS pedidos
    FROM pedidos p
    JOIN itens_pedido i       ON p.order_id = i.order_id
    JOIN produtos pr          ON i.product_id = pr.product_id
    LEFT JOIN categorias_en c ON pr.product_category_name = c.product_category_name
    WHERE p.order_status = 'delivered'
    GROUP BY categoria
),
ranking AS (
    SELECT categoria,
           ROUND(receita, 2)                                AS receita,
           pedidos,
           ROUND(100.0 * receita / SUM(receita) OVER (), 1) AS participacao_pct,
           DENSE_RANK() OVER (ORDER BY receita DESC)        AS posicao
    FROM receita_categoria
)
SELECT * FROM ranking WHERE posicao <= 10 ORDER BY posicao   -- janela não pode ir no WHERE da mesma consulta
''')

,categoria,receita,pedidos,participacao_pct,posicao
0,health_beauty,1412271.15,8647,9.2,1
1,watches_gifts,1264333.12,5495,8.2,2
2,bed_bath_table,1225209.26,9272,7.9,3
3,sports_leisure,1119923.63,7530,7.3,4
4,computers_accessories,1032723.77,6530,6.7,5
5,furniture_decor,881063.91,6307,5.7,6
6,housewares,759313.29,5743,4.9,7
7,cool_stuff,691680.89,3559,4.5,8
8,auto,669559.87,3810,4.3,9
9,garden_tools,567354.11,3448,3.7,10


**Conclusão:** as 10 maiores categorias concentram 62,4% da receita. A líder é health_beauty (9,2%). watches_gifts é a 2ª em receita com bem menos pedidos que bed_bath_table (5.495 contra 9.272), sinal de ticket médio mais alto.

## 4. Ticket médio por estado do cliente
*Conceitos:* JOIN `clientes` × `pedidos` × `pagamentos`, `GROUP BY`, `HAVING` para tirar estados com poucos pedidos.

In [25]:
q('''
SELECT c.customer_state                                       AS estado,
       COUNT(DISTINCT p.order_id)                             AS pedidos,
       ROUND(SUM(pg.payment_value), 2)                        AS receita,
       ROUND(SUM(pg.payment_value) / COUNT(DISTINCT p.order_id), 2) AS ticket_medio
FROM clientes c
JOIN pedidos p     ON c.customer_id = p.customer_id
JOIN pagamentos pg ON p.order_id   = pg.order_id      -- um pedido pode ter várias linhas de pagamento
WHERE p.order_status = 'delivered'                    -- filtra LINHAS (antes de agrupar)
GROUP BY c.customer_state
HAVING COUNT(DISTINCT p.order_id) >= 100              -- filtra GRUPOS (depois de agrupar)
ORDER BY ticket_medio DESC
''')

,estado,pedidos,receita,ticket_medio
0,PB,517,137834.65,266.60
1,AL,397,94195.79,237.27
2,RO,243,56975.70,234.47
3,PA,946,212027.55,224.13
4,PI,476,105272.17,221.16
5,TO,274,60007.37,219.00
6,RN,474,100728.30,212.51
7,SE,335,70289.13,209.82
8,CE,1279,266463.97,208.34
9,MA,717,147807.29,206.15


**Conclusão:** o maior ticket médio está na Paraíba (R$ 266,60) e o menor em São Paulo (R$ 142,48), que concentra o maior volume (40.500 pedidos). Estados do Norte e Nordeste aparecem no topo do ranking de ticket; uma hipótese a testar é o peso do frete mais caro para regiões distantes dos vendedores.

## 5. Atraso na entrega por estado
% de pedidos entregues depois da data estimada (`order_delivered_customer_date > order_estimated_delivery_date`).

*Conceitos:* `CASE WHEN`, média de indicador 0/1.

In [26]:
q('''
SELECT c.customer_state AS estado,
       COUNT(*)         AS pedidos_entregues,
       SUM(CASE WHEN date(p.order_delivered_customer_date) > date(p.order_estimated_delivery_date)
                THEN 1 ELSE 0 END) AS pedidos_atrasados,
       ROUND(100.0 * AVG(CASE WHEN date(p.order_delivered_customer_date) > date(p.order_estimated_delivery_date)
                              THEN 1 ELSE 0 END), 1) AS pct_atraso
FROM pedidos p
JOIN clientes c ON p.customer_id = c.customer_id
WHERE p.order_status = 'delivered'
  AND p.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
HAVING COUNT(*) >= 100
ORDER BY pct_atraso DESC
''')

,estado,pedidos_entregues,pedidos_atrasados,pct_atraso
0,AL,397,85,21.4
1,MA,717,125,17.4
2,SE,335,51,15.2
3,PI,476,66,13.9
4,CE,1279,176,13.8
5,BA,3256,396,12.2
6,RJ,12350,1495,12.1
7,PA,946,106,11.2
8,ES,1995,214,10.7
9,PB,517,54,10.4


**Conclusão:** 6,8% dos pedidos entregues chegaram depois da data estimada. Os piores estados são Alagoas (21,4%) e Maranhão (17,4%); os melhores são Amazonas (2,8%), Rondônia (2,9%) e Paraná (4,0%). O Rio de Janeiro merece atenção: 12,1% de atraso sobre um volume alto (1.495 pedidos atrasados).

## 6. Atraso derruba a nota?
Nota média de avaliação por faixa de atraso: no prazo, 1–3 dias, 4–7 dias, mais de 7 dias.

*Conceitos:* `julianday()` para diferença de datas, `CASE` para faixas.

In [27]:
q('''
WITH atraso AS (
    SELECT order_id,
           julianday(date(order_delivered_customer_date))
         - julianday(date(order_estimated_delivery_date)) AS dias_atraso   -- negativo = chegou antes
    FROM pedidos
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
),
nota AS (SELECT order_id, AVG(review_score) AS nota FROM avaliacoes GROUP BY order_id),
base AS (
    SELECT n.nota,
           CASE WHEN a.dias_atraso <= 0 THEN '1. No prazo'
                WHEN a.dias_atraso <= 3 THEN '2. 1 a 3 dias'
                WHEN a.dias_atraso <= 7 THEN '3. 4 a 7 dias'
                ELSE                         '4. Mais de 7 dias' END AS faixa_atraso
    FROM atraso a JOIN nota n ON a.order_id = n.order_id
)
SELECT faixa_atraso,
       COUNT(*)                                                     AS pedidos,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)           AS pct_pedidos,
       ROUND(AVG(nota), 2)                                          AS nota_media,
       ROUND(100.0 * AVG(CASE WHEN nota <= 2 THEN 1 ELSE 0 END), 1) AS pct_notas_1_e_2
FROM base
GROUP BY faixa_atraso
ORDER BY faixa_atraso
''')

,faixa_atraso,pedidos,pct_pedidos,nota_media,pct_notas_1_e_2
0,1. No prazo,89443,93.3,4.29,9.2
1,2. 1 a 3 dias,1852,1.9,3.29,32.2
2,3. 4 a 7 dias,1748,1.8,2.11,67.6
3,4. Mais de 7 dias,2781,2.9,1.70,79.2


**Conclusão:** sim. Pedidos no prazo têm nota média 4,29; com mais de 7 dias de atraso a nota cai para 1,70 e 79,2% dos clientes dão nota 1 ou 2. O atraso atinge poucos pedidos, mas está fortemente associado à insatisfação (associação, não prova de causa).

## 7. Taxa de recompra
Quantos clientes únicos (`customer_unique_id`) fizeram mais de um pedido?

*Conceitos:* subconsulta ou CTE com `COUNT(DISTINCT order_id)`.

In [28]:
# Período da base
display(q('''
SELECT MIN(date(order_purchase_timestamp)) AS primeira_compra,
       MAX(date(order_purchase_timestamp)) AS ultima_compra
FROM pedidos
'''))

# Taxa de recompra: clientes com 2+ pedidos ÷ total (por PESSOA = customer_unique_id)
q('''
WITH pedidos_por_cliente AS (
    SELECT c.customer_unique_id, COUNT(DISTINCT p.order_id) AS qtd_pedidos
    FROM clientes c
    JOIN pedidos p ON c.customer_id = p.customer_id
    WHERE p.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)
SELECT COUNT(*)                                          AS clientes_total,
       SUM(CASE WHEN qtd_pedidos >= 2 THEN 1 ELSE 0 END) AS clientes_recompra,
       ROUND(100.0 * AVG(CASE WHEN qtd_pedidos >= 2 THEN 1 ELSE 0 END), 2) AS taxa_recompra_pct
FROM pedidos_por_cliente
''')

,primeira_compra,ultima_compra
0,2016-09-04,2018-10-17


,clientes_total,clientes_recompra,taxa_recompra_pct
0,93358,2801,3.0


In [29]:
# Tempo entre uma compra e a seguinte do mesmo cliente (LAG + PARTITION BY)
q('''
WITH compras AS (
    SELECT c.customer_unique_id,
           date(p.order_purchase_timestamp) AS data_compra,
           LAG(date(p.order_purchase_timestamp)) OVER (
               PARTITION BY c.customer_unique_id ORDER BY p.order_purchase_timestamp) AS compra_anterior
    FROM clientes c
    JOIN pedidos p ON c.customer_id = p.customer_id
    WHERE p.order_status = 'delivered'
)
SELECT CASE WHEN julianday(data_compra) - julianday(compra_anterior) = 0   THEN '0. Mesmo dia'
            WHEN julianday(data_compra) - julianday(compra_anterior) <= 30  THEN '1. 1 a 30 dias'
            WHEN julianday(data_compra) - julianday(compra_anterior) <= 180 THEN '2. 31 a 180 dias'
            ELSE '3. Mais de 180 dias' END AS intervalo_recompra,
       COUNT(*) AS recompras,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM compras
WHERE compra_anterior IS NOT NULL
GROUP BY intervalo_recompra
ORDER BY intervalo_recompra
''')

,intervalo_recompra,recompras,pct
0,0. Mesmo dia,890,28.5
1,1. 1 a 30 dias,678,21.7
2,2. 31 a 180 dias,1043,33.4
3,3. Mais de 180 dias,509,16.3


**Conclusão:** a base vai de set/2016 a out/2018. Só 3,0% dos clientes (2.801 de 93.358) fizeram mais de um pedido entregue: a Olist depende quase toda de clientes novos. Entre as recompras, 28,5% acontecem no mesmo dia (provavelmente pedidos separados na mesma visita) e 33,4% entre 31 e 180 dias.

## 8. Coorte de primeira compra
Para cada mês de primeira compra, quantos clientes voltaram nos meses seguintes?

*Conceitos:* `MIN()` por cliente em uma CTE, JOIN de volta com pedidos.

In [30]:
coorte = q('''
WITH compras AS (
    SELECT c.customer_unique_id, date(p.order_purchase_timestamp) AS data_compra
    FROM clientes c JOIN pedidos p ON c.customer_id = p.customer_id
    WHERE p.order_status = 'delivered'
),
primeira AS (                                                -- a coorte de cada cliente
    SELECT customer_unique_id, MIN(data_compra) AS primeira_compra
    FROM compras GROUP BY customer_unique_id
),
base AS (
    SELECT c.customer_unique_id,
           strftime('%Y-%m', f.primeira_compra) AS coorte,
           (CAST(strftime('%Y', c.data_compra) AS INT) - CAST(strftime('%Y', f.primeira_compra) AS INT)) * 12
         + (CAST(strftime('%m', c.data_compra) AS INT) - CAST(strftime('%m', f.primeira_compra) AS INT)) AS meses_depois
    FROM compras c JOIN primeira f ON c.customer_unique_id = f.customer_unique_id
)
SELECT coorte, meses_depois, COUNT(DISTINCT customer_unique_id) AS clientes
FROM base
GROUP BY coorte, meses_depois
''')

# Tabela em escada: % da turma que voltou em cada mês (só coortes de 2017 em diante, até 6 meses)
escada   = coorte.pivot(index='coorte', columns='meses_depois', values='clientes')
retencao = (escada.div(escada[0], axis=0) * 100).round(2)
retencao.loc['2017-01':'2018-08', 0:6].style.background_gradient(cmap='Blues', axis=None, subset=list(range(1, 7))).format('{:.2f}%', na_rep='')

meses_depois,0,1,2,3,4,5,6
coorte,,,,,,,
2017-01,100.00%,0.28%,0.28%,0.14%,0.42%,0.14%,0.42%
2017-02,100.00%,0.18%,0.31%,0.12%,0.43%,0.12%,0.25%
2017-03,100.00%,0.44%,0.36%,0.40%,0.36%,0.16%,0.16%
2017-04,100.00%,0.62%,0.22%,0.18%,0.27%,0.27%,0.35%
2017-05,100.00%,0.46%,0.46%,0.29%,0.29%,0.32%,0.41%
2017-06,100.00%,0.49%,0.40%,0.43%,0.30%,0.40%,0.36%
2017-07,100.00%,0.53%,0.35%,0.24%,0.29%,0.21%,0.32%
2017-08,100.00%,0.69%,0.35%,0.27%,0.35%,0.52%,0.30%
2017-09,100.00%,0.70%,0.55%,0.27%,0.45%,0.22%,0.22%


**Conclusão:** em todas as coortes de 2017-01 a 2018-07, menos de 1% da turma volta no mês seguinte (entre 0,18% e 0,72%, média de 0,48%), e a retenção segue abaixo de 1% nos meses seguintes. Nenhuma safra se destaca, então o problema de retenção é estrutural, o que reforça a pergunta 7 e justifica a segmentação.

## 9. Formas de pagamento
Participação de cada `payment_type` no valor total e número médio de parcelas.

In [31]:
q('''
SELECT pg.payment_type                                    AS tipo_pagamento,
       COUNT(DISTINCT pg.order_id)                        AS pedidos,
       ROUND(SUM(pg.payment_value), 2)                    AS valor_total,
       ROUND(100.0 * SUM(pg.payment_value)
             / SUM(SUM(pg.payment_value)) OVER (), 1)     AS participacao_pct,
       ROUND(AVG(pg.payment_installments), 1)             AS parcelas_media
FROM pagamentos pg
JOIN pedidos p ON pg.order_id = p.order_id
WHERE p.order_status = 'delivered'
GROUP BY pg.payment_type
ORDER BY valor_total DESC
''')

,tipo_pagamento,pedidos,valor_total,participacao_pct,parcelas_media
0,credit_card,74304,12101094.88,78.5,3.5
1,boleto,19191,2769932.58,18.0,1.0
2,voucher,3679,343013.19,2.2,1.0
3,debit_card,1485,208421.12,1.4,1.0


**Conclusão:** o cartão de crédito concentra 78,5% do valor pago, com média de 3,5 parcelas; o boleto responde por 18,0%. Voucher e débito somam 3,6% e são pagos à vista. O parcelamento no cartão é central para o modelo de negócio.

## 10. Tabela RFM por cliente *(vai alimentar a segmentação)*
Para cada `customer_unique_id`, considerando só pedidos entregues:
- **Recência:** dias entre a última compra do cliente e a última data da base
- **Frequência:** número de pedidos
- **Valor:** soma paga

Salve o resultado em uma tabela nova: `CREATE TABLE rfm AS ...`

In [ ]:
con.execute("DROP TABLE IF EXISTS rfm")
con.execute('''
CREATE TABLE rfm AS
WITH pago AS (                                   -- valor pago por pedido
    SELECT order_id, SUM(payment_value) AS valor_pedido FROM pagamentos GROUP BY order_id
),
base AS (
    SELECT c.customer_unique_id, p.order_id,
           date(p.order_purchase_timestamp) AS data_compra, pg.valor_pedido
    FROM clientes c
    JOIN pedidos p ON c.customer_id = p.customer_id
    JOIN pago pg   ON p.order_id   = pg.order_id
    WHERE p.order_status = 'delivered'
),
referencia AS (SELECT MAX(data_compra) AS ultima_data_base FROM base)   -- data de referência
SELECT b.customer_unique_id,
       CAST(julianday(r.ultima_data_base) - julianday(MAX(b.data_compra)) AS INT) AS recencia,
       COUNT(DISTINCT b.order_id)    AS frequencia,
       ROUND(SUM(b.valor_pedido), 2) AS valor
FROM base b CROSS JOIN referencia r
GROUP BY b.customer_unique_id
''')
con.commit()

q('''
SELECT COUNT(*) AS clientes,
       MIN(recencia)   AS rec_min,  ROUND(AVG(recencia), 0)   AS rec_media,  MAX(recencia)   AS rec_max,
       MIN(frequencia) AS freq_min, ROUND(AVG(frequencia), 2) AS freq_media, MAX(frequencia) AS freq_max,
       MIN(valor)      AS val_min,  ROUND(AVG(valor), 2)      AS val_media,  MAX(valor)      AS val_max
FROM rfm
''')

**Conclusão:** a tabela RFM tem 93.357 clientes únicos (um a menos que na pergunta 7 porque um pedido entregue não tem registro de pagamento). Recência média de 237 dias (0 a 695), frequência média de 1,03 pedido e valor médio de R$ 165,20, com máximo de R$ 13.664,08. A forte assimetria do valor pede transformação logarítmica antes da clusterização.